In [174]:
# Import Packages
import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import os
from groq import Groq
from dotenv import dotenv_values
from pprint import pprint
from google import genai
from google.genai import types
from pypdf import PdfReader
import requests
from groq import Groq
from sentence_transformers import SentenceTransformer
import re
import pdfplumber
from rank_bm25 import BM25Okapi

In [153]:
# Import Varaibles
config = dotenv_values(".env")
GROQ_API_KEY = config["GROQ_API_KEY"]
GEMENI_API_KEY = config["GEMENI_API_KEY"]
HF_TOKEN =config.get("HF_TOKEN")

In [154]:
# Import Embedding Model and LLM Model
os.environ["HF_TOKEN"] = HF_TOKEN
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load LLM Model
# ollama server --> this will run model in locally in http://localhost:11434

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2619.99it/s]


In [155]:
# try all-MiniLM-L6-v2 Model
sentences = "What is Meaning Of Life ?"
embeddings = model.encode(sentences)
print(embeddings.shape)

(384,)


In [156]:
# Init Chroma Vector Database
chroma_client = chromadb.PersistentClient(path="course")
collecation_name = "course"
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(name=collecation_name, embedding_function=embedding_function)
print(collection)

Collection(name=course)


In [157]:
documents = []
with pdfplumber.open("xml_course.pdf") as pdf:
    print(f"number of pages is: {len(pdf.pages)}")

    for i, page in enumerate(pdf.pages):
        text = page.extract_text()

        if text:
            # remove extra spaces
            text = text.strip()

            # fix line breaks inside paragraphs
            text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

            # normalize multiple spaces/newlines
            text = re.sub(r'\s+', ' ', text)

            documents.append({"id": i, "text": text})
        else:
            documents.append("")

print("number of documents: ", len(documents))

number of pages is: 55
number of documents:  55


In [158]:
# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [159]:
# Split documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(doc["text"]):
        chunked_documents.append({"id": f"{doc["id"]}_chunk{i+1}", "text": chunk})
print(len(chunked_documents))

62947


In [160]:
# Function to Generate embeddings for the document chunks - Doc2Vect
def get_embedding(text):
    embeddings = model.encode(sentences)
    return embeddings

In [161]:
# Get Embedding Vectors
chunked_documents = chunked_documents[0: 1000]
for doc_chunk in chunked_documents:
    doc_chunk['embeddings'] = get_embedding(doc_chunk["text"])


In [162]:
# Save Embedding Into Vector Database
for doc_chunk in chunked_documents:
    collection.upsert(ids=[doc_chunk["id"]], documents=[doc_chunk["text"]], embeddings=[doc_chunk["embeddings"]])


In [163]:
# Get Documents from LLM
client_groq = Groq(api_key=GROQ_API_KEY)
def get_documents(question):
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role": "system",
            "content": f"You are a technical documentation writer. Write 3 clear\
            structured documentation for: {question}",
        },
        ],
        temperature=1,
        max_completion_tokens=1024, 
        top_p=1,
        stream=True,
        stop=None
    )

    res = []
    for chunk in completion:
        res.append(chunk.choices[0].delta.content)

    res = [s for s in res if s]
    res = "".join(res)
    return res

In [164]:
# get LLM Documents
response = get_documents(question="What is AI ?")

In [165]:
# Function to Split LLM Document
def split_text_llm(text):
    chunks = []

    for i, p in enumerate(text.split("\n")):
        p = p.strip()

        if len(p) > 20:
            chunks.append({
                "id": f"chunk_{i}",
                "text": p,
                "embedding": None
            })

    return chunks

In [166]:
llm_documents = split_text_llm(response)

In [167]:
# Get Chunks from LLM Documents
llm_documents_chunks = []
for doc in llm_documents:
    chunks = split_text_llm(doc["text"])
    llm_documents_chunks.append({"text": chunks})

In [168]:
# Embedding LLM Document
for doc_chunk in llm_documents_chunks:
    doc_chunk['embeddings'] = get_embedding(doc_chunk["text"])

In [180]:
# Function That Query Documents
'''
    collection.query(): find the documents most related to text
        - query_texts: the question or text you want to search with.
        - n_results: how many documents we want back.
'''
corpus = [doc["text"].split() for doc in documents]
bm25 = BM25Okapi(corpus)
def bm25_search(question, chunked_documents, top_k=2):
    tokenized_query = question.split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    return [chunked_documents[i] for i in top_indices]

print(bm25_search("What is XML ?", chunked_documents))

[{'id': '0_chunk40', 'text': 'r', 'embeddings': array([-4.89503406e-02,  9.48239043e-02, -6.59489632e-02,  1.60414111e-02,
       -4.12474610e-02,  8.13594181e-03,  1.03465356e-01, -1.59825943e-02,
        8.78064111e-02,  5.87436184e-03,  3.05233523e-02, -5.84755912e-02,
       -2.66282037e-02,  5.26752789e-04, -6.66964101e-03, -3.30435112e-02,
       -6.74314499e-02, -2.24154610e-02, -1.40454648e-02, -1.58117078e-02,
        3.09546869e-02,  1.03859613e-02, -3.21427621e-02,  6.26181141e-02,
       -9.48482156e-02,  1.06964357e-01, -6.52563572e-03,  2.64959428e-02,
        9.69061395e-04,  1.90223269e-02,  2.79137883e-02,  5.97960241e-02,
        9.16541293e-02, -2.45239213e-02, -1.40005210e-03,  4.20360453e-02,
        1.00736178e-01, -2.66225096e-02,  1.30868731e-02,  1.37929125e-02,
       -4.37666178e-02, -1.91826168e-02,  5.06948587e-03, -1.84548309e-03,
        4.25119922e-02, -4.99994541e-03,  2.00772211e-02, -8.56257677e-02,
        1.30573541e-01,  1.51056992e-02, -5.59229068

In [177]:
# 
print(chunked_documents[0])
print(llm_documents[0])

{'id': '0_chunk1', 'text': 'X', 'embeddings': array([-4.89503406e-02,  9.48239043e-02, -6.59489632e-02,  1.60414111e-02,
       -4.12474610e-02,  8.13594181e-03,  1.03465356e-01, -1.59825943e-02,
        8.78064111e-02,  5.87436184e-03,  3.05233523e-02, -5.84755912e-02,
       -2.66282037e-02,  5.26752789e-04, -6.66964101e-03, -3.30435112e-02,
       -6.74314499e-02, -2.24154610e-02, -1.40454648e-02, -1.58117078e-02,
        3.09546869e-02,  1.03859613e-02, -3.21427621e-02,  6.26181141e-02,
       -9.48482156e-02,  1.06964357e-01, -6.52563572e-03,  2.64959428e-02,
        9.69061395e-04,  1.90223269e-02,  2.79137883e-02,  5.97960241e-02,
        9.16541293e-02, -2.45239213e-02, -1.40005210e-03,  4.20360453e-02,
        1.00736178e-01, -2.66225096e-02,  1.30868731e-02,  1.37929125e-02,
       -4.37666178e-02, -1.91826168e-02,  5.06948587e-03, -1.84548309e-03,
        4.25119922e-02, -4.99994541e-03,  2.00772211e-02, -8.56257677e-02,
        1.30573541e-01,  1.51056992e-02, -5.59229068e-